# Train YOLOv8 nano model

In [2]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(data = "data.yaml", imgsz=640, batch=8, epochs=100, workers=4, device=0)

New https://pypi.org/project/ultralytics/8.4.128 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.117  Python-3.14.2 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000029BEC97CD70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.04

# Test YOLOv8 nano custom model

In [1]:
from pathlib import Path
from ultralytics import YOLO
import cv2
import threading
from queue import Queue

model = YOLO("yolov8n_custom.pt")

# Video capture with threading for non-blocking reading
class FrameCapture:
    def __init__(self, source=0, fps_limit=30):
        self.cap = cv2.VideoCapture(source)
        self.cap.set(cv2.CAP_PROP_FPS, fps_limit)
        self.frame = None
        self.running = True
        self.thread = threading.Thread(target=self.read_frames, daemon=True)
        self.thread.start()
    
    def read_frames(self):
        while self.running:
            ret, frame = self.cap.read()
            if ret:
                self.frame = frame
    
    def get_frame(self):
        return self.frame
    
    def release(self):
        self.running = False
        self.cap.release()

# Start capture with FPS limit
frame_reader = FrameCapture(source=0, fps_limit=30)

print("Starting webcam detection...")
print("Press 'q' to quit")

frame_count = 0
skip_frames = 2  # Process every 3rd frame for speed

try:
    while True:
        frame = frame_reader.get_frame()
        if frame is None:
            continue
        
        frame_count += 1
        
        # Skip frames for real-time performance
        if frame_count % (skip_frames + 1) != 0:
            continue
        
        # Resize for faster inference (optional)
        resized = cv2.resize(frame, (640, 480))
        
        # Predict with confidence threshold
        results = model.predict(
            source=resized,
            device=0,
            conf=0.5,  # Only detections > 50% confidence
            imgsz=640,
            verbose=False
        )
        
        # Draw results
        for r in results:
            frame = r.plot()
        
        cv2.imshow("YOLO Detection", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    frame_reader.release()
    cv2.destroyAllWindows()
    print("Webcam detection stopped")

ModuleNotFoundError: No module named 'ultralytics'

## Predict Using YOLOv8n_custom (image and label in test folder)

In [7]:
from pathlib import Path
import csv
from collections import Counter
from ultralytics import YOLO

model = YOLO("yolov8n_custom.pt")
test_images = Path("test/images")
test_labels = Path("test/labels")
output_dir = Path("runs/gesture_test")
output_dir.mkdir(parents=True, exist_ok=True)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_paths = sorted(
    path for path in test_images.iterdir()
    if path.suffix.lower() in image_extensions
)

if not image_paths:
    raise FileNotFoundError(f"No images found in {test_images.resolve()}")

# Compare one gesture per image: the highest-confidence prediction vs. its label.
rows = []
correct = 0
class_names = model.names

for image_path in image_paths:
    label_path = test_labels / f"{image_path.stem}.txt"
    expected_ids = []
    if label_path.exists():
        with label_path.open(encoding="utf-8") as label_file:
            expected_ids = [int(line.split()[0]) for line in label_file if line.strip()]

    predictions = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        device=0,
        verbose=False,
        save=True,
        project=str(output_dir),
        name="annotated",
        exist_ok=True,
    )[0]

    if predictions.boxes is not None and len(predictions.boxes):
        best_index = int(predictions.boxes.conf.argmax().item())
        predicted_id = int(predictions.boxes.cls[best_index].item())
        confidence = float(predictions.boxes.conf[best_index].item())
    else:
        predicted_id = None
        confidence = 0.0

    expected_id = expected_ids[0] if expected_ids else None
    is_correct = predicted_id == expected_id and predicted_id is not None
    correct += int(is_correct)

    rows.append({
        "image": image_path.name,
        "expected": class_names.get(expected_id, str(expected_id)) if expected_id is not None else "missing label",
        "predicted": class_names.get(predicted_id, str(predicted_id)) if predicted_id is not None else "no detection",
        "confidence": f"{confidence:.4f}",
        "result": "RIGHT" if is_correct else "WRONG",
    })

report_path = output_dir / "gesture_results.csv"
with report_path.open("w", newline="", encoding="utf-8") as report_file:
    writer = csv.DictWriter(report_file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

result_counts = Counter(row["result"] for row in rows)
accuracy = correct / len(rows)
print(f"Test images: {len(rows)}")
print(f"Right:       {result_counts['RIGHT']}")
print(f"Wrong:       {result_counts['WRONG']}")
print(f"Accuracy:    {accuracy:.2%}")
print(f"CSV report:  {report_path.resolve()}")
print(f"Annotated:   {(output_dir / 'annotated').resolve()}")

for row in rows:
    print(f"{row['result']:5} | {row['image']} | expected={row['expected']} | predicted={row['predicted']} | confidence={row['confidence']}")


Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\annotated
Results saved to C:\Users\ah_pe\runs\detect\runs\gesture_test\an